# Pipeline QA génératif — Interface Web complète (Gradio)

Interface web à 2 onglets :
1. **Générer le dataset** — upload un document (PDF/DOCX/TXT/CSV/image/JSON), la pipeline l'ingère, le découpe, et génère automatiquement des paires question/réponse.
2. **Entraîner un modèle** — choisis un modèle et une méthode de fine-tuning, lance le traitement.

Tout est sauvegardé automatiquement sur Google Drive à chaque étape.

## Étape 0 — GPU + Google Drive

In [ ]:
import torch
print("GPU disponible :", torch.cuda.is_available())
print("Nom du GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun")
print("VRAM :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "Go")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
SAVE_DIR = "/content/drive/MyDrive/projet_qa_stage"
os.makedirs(SAVE_DIR, exist_ok=True)

def load_from_drive(filenames):
    recovered = []
    for f in filenames:
        src = os.path.join(SAVE_DIR, f)
        if os.path.exists(src):
            shutil.copy(src, f)
            recovered.append(f)
    print(f"Récupérés depuis Drive : {recovered}")

def save_to_drive(filenames):
    for f in filenames:
        if os.path.exists(f):
            shutil.copy(f, os.path.join(SAVE_DIR, f))
    print(f"Sauvegardé sur Drive : {filenames}")

def save_dir_to_drive(local_dir):
    # Version "dossier" (pour les modèles entraînés, qui contiennent plusieurs fichiers)
    if os.path.exists(local_dir):
        shutil.copytree(local_dir, os.path.join(SAVE_DIR, local_dir), dirs_exist_ok=True)
        print(f"Sauvegardé sur Drive : {local_dir}/")

def load_dir_from_drive(local_dir):
    src = os.path.join(SAVE_DIR, local_dir)
    if os.path.exists(src):
        shutil.copytree(src, local_dir, dirs_exist_ok=True)
        print(f"Récupéré depuis Drive : {local_dir}/")

# Tente de récupérer un dataset déjà généré lors d'une session précédente (optionnel)
load_from_drive(["train.jsonl", "val.jsonl", "test.jsonl"])
# Tente de récupérer d'éventuels modèles déjà entraînés lors d'une session précédente
load_dir_from_drive("trained_models")

## Étape 1 — Installer les librairies

In [ ]:
!pip install -q \
    "transformers==5.15.1" "trl==1.10.0" "peft==0.20.0" "accelerate==1.14.0" "datasets==5.0.1" \
    "bitsandbytes>=0.45.0" evaluate rouge_score bert_score \
    sentencepiece gradio pypdf pytesseract pillow python-docx pandas
!apt-get -qq install -y tesseract-ocr tesseract-ocr-fra > /dev/null

import peft.tuners.lora.torchao as torchao_check_module
torchao_check_module.is_torchao_available = lambda: False
print("✅ Librairies prêtes (versions figées pour éviter les cassures d'API).")

## Étape 2 — Fonctions d'ingestion (chargeur universel + chunking générique)

In [ ]:
import re, json, os
from pypdf import PdfReader
from PIL import Image
import pytesseract
import docx
import pandas as pd


def load_pdf(path):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

def load_image(path):
    return pytesseract.image_to_string(Image.open(path), lang="fra")

def load_txt(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def load_json_doc(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    texts = []
    def extract_strings(obj):
        if isinstance(obj, str):
            if len(obj.strip()) > 3:
                texts.append(obj.strip())
        elif isinstance(obj, dict):
            for v in obj.values():
                extract_strings(v)
        elif isinstance(obj, list):
            for item in obj:
                extract_strings(item)
    extract_strings(data)
    return "\n".join(texts)

def load_docx(path):
    document = docx.Document(path)
    return "\n".join(p.text for p in document.paragraphs if p.text.strip())

def load_csv_doc(path):
    df = pd.read_csv(path)
    text_columns = df.select_dtypes(include="object").columns
    texts = []
    for col in text_columns:
        texts.extend(df[col].dropna().astype(str).tolist())
    return "\n".join(texts)

def load_document(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf": return load_pdf(path)
    elif ext in [".png", ".jpg", ".jpeg", ".bmp", ".tiff"]: return load_image(path)
    elif ext == ".txt": return load_txt(path)
    elif ext == ".json": return load_json_doc(path)
    elif ext == ".docx": return load_docx(path)
    elif ext == ".csv": return load_csv_doc(path)
    else: raise ValueError(f"Format non supporté : {ext}")


def clean_text(text):
    lines = text.split("\n")
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in lines]
    return "\n".join([l for l in lines if l])

def split_into_paragraphs(text):
    raw_lines = [l for l in text.split("\n") if l.strip()]
    paragraphs, buffer = [], ""
    for line in raw_lines:
        buffer = (buffer + " " + line).strip() if buffer else line
        if re.search(r"[.!?:]$", line):
            paragraphs.append(buffer)
            buffer = ""
    if buffer:
        paragraphs.append(buffer)
    return paragraphs

def is_probable_heading(paragraph):
    words = paragraph.split()
    if len(words) == 0 or len(words) > 8:
        return False
    if re.search(r"[.!?]$", paragraph):
        return False
    letters = [c for c in paragraph if c.isalpha()]
    if not letters:
        return False
    upper_ratio = sum(1 for c in letters if c.isupper()) / len(letters)
    return upper_ratio > 0.5 or paragraph == paragraph.title()

def split_into_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-ZÀ-Ú])', text)
    return [s.strip() for s in sentences if len(s.strip()) > 0]

def chunk_by_paragraph_or_topic(text, target_words=250, max_words=400, overlap_paragraphs=1):
    paragraphs = split_into_paragraphs(text)
    normalized = []
    for p in paragraphs:
        if len(p.split()) > max_words:
            normalized.extend(split_into_sentences(p))
        else:
            normalized.append(p)

    chunks_out, current, count, idx = [], [], 0, 0

    def flush():
        nonlocal current, count, idx
        if current:
            chunks_out.append({"chunk_id": f"chunk_{idx:03d}", "text": "\n".join(current), "n_words": count})
            idx += 1

    for p in normalized:
        n_words = len(p.split())
        if is_probable_heading(p) and count >= 30:
            flush()
            current = current[-overlap_paragraphs:] if overlap_paragraphs else []
            count = sum(len(x.split()) for x in current)
        current.append(p)
        count += n_words
        if count >= target_words:
            flush()
            current = current[-overlap_paragraphs:] if overlap_paragraphs else []
            count = sum(len(x.split()) for x in current)
    if count >= 30:
        flush()
    return chunks_out

print("✅ Fonctions d'ingestion prêtes.")

## Étape 3 — Fonctions de génération Q/A (Qwen2.5-3B-Instruct)

In [ ]:
PROMPT_TEMPLATE = """Voici un extrait d'un document :

---
{chunk_text}
---

Génère exactement 6 paires question/réponse en français, basées UNIQUEMENT sur cet extrait.
Varie impérativement les 6 types de questions suivants (une de chaque) :
1. Une question factuelle directe
2. Une question de synthèse/reformulation du principe général
3. Une question sur un cas pratique ou une mise en situation
4. Une question qui commence par "Pourquoi..." (justification/raison d'être)
5. Une question de type "Que se passe-t-il si..." (conséquence)
6. Une question destinée à quelqu'un qui découvre ce contenu pour la première fois

Les réponses doivent être complètes mais concises (2-4 phrases), reformulées avec tes propres mots.

Réponds UNIQUEMENT avec un tableau JSON valide, sans aucun texte avant ou après, au format :
[{{"question": "...", "answer": "..."}}, ...]
"""

_gen_model = None
_gen_tokenizer = None

def load_generation_model():
    global _gen_model, _gen_tokenizer
    if _gen_model is None:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        GEN_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
        _gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
        _gen_model = AutoModelForCausalLM.from_pretrained(GEN_MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto")
    return _gen_model, _gen_tokenizer

def unload_generation_model():
    global _gen_model, _gen_tokenizer
    import gc
    _gen_model = None
    _gen_tokenizer = None
    gc.collect()
    torch.cuda.empty_cache()

def generate_qa(chunk_text, max_new_tokens=1000):
    model, tokenizer = load_generation_model()
    messages = [{"role": "user", "content": PROMPT_TEMPLATE.format(chunk_text=chunk_text)}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.7,
                                  do_sample=True, pad_token_id=tokenizer.eos_token_id)
    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    match = re.search(r"\[.*\]", generated, re.DOTALL)
    if not match:
        return []
    try:
        raw = json.loads(match.group(0))
        return [qa for qa in raw if isinstance(qa, dict) and "question" in qa and "answer" in qa]
    except json.JSONDecodeError:
        return []

print("✅ Fonctions de génération Q/A prêtes.")

## Étape 4 — Pipeline complet de création du dataset (générateur avec suivi de progression)

In [ ]:
import random

def is_valid_qa(row):
    q, a = row["question"].strip(), row["answer"].strip()
    return len(q) >= 10 and len(a) >= 15 and q.lower() != a.lower() and len(a.split()) <= 150

def build_dataset_pipeline(doc_path):
    log = f"📄 Document reçu : {os.path.basename(doc_path)}\n"
    yield log, None

    raw_text = clean_text(load_document(doc_path))
    chunks = chunk_by_paragraph_or_topic(raw_text, target_words=250, max_words=400, overlap_paragraphs=1)
    log += f"✂️ {len(chunks)} chunks générés ({len(raw_text.split())} mots au total)\n"
    yield log, None

    dataset = []
    for i, chunk in enumerate(chunks):
        qa_pairs = []
        for attempt in range(3):
            qa_pairs = generate_qa(chunk["text"])
            if qa_pairs:
                break
        for qa in qa_pairs:
            dataset.append({"chunk_id": chunk["chunk_id"], "context": chunk["text"],
                             "question": str(qa.get("question", "")), "answer": str(qa.get("answer", ""))})
        if (i + 1) % 5 == 0 or i == len(chunks) - 1:
            log += f"🤖 Génération Q/A : chunk {i+1}/{len(chunks)} — {len(dataset)} paires au total\n"
            yield log, None

    unload_generation_model()

    clean_dataset = [row for row in dataset if is_valid_qa(row)]
    seen, deduped = set(), []
    for row in clean_dataset:
        qn = row["question"].strip().lower()
        if qn not in seen:
            seen.add(qn)
            deduped.append(row)

    random.seed(42)
    random.shuffle(deduped)
    n = len(deduped)
    train, val, test = deduped[:int(0.8*n)], deduped[int(0.8*n):int(0.9*n)], deduped[int(0.9*n):]

    for name, split in [("train", train), ("val", val), ("test", test)]:
        with open(f"{name}.jsonl", "w", encoding="utf-8") as f:
            for row in split:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

    save_to_drive(["train.jsonl", "val.jsonl", "test.jsonl"])

    log += (
        f"\n✅ Dataset prêt !\n"
        f"Brut : {len(dataset)} | Après nettoyage : {len(clean_dataset)} | Après dédoublonnage : {len(deduped)}\n"
        f"Train : {len(train)} | Val : {len(val)} | Test : {len(test)}\n"
        f"Sauvegardé sur Drive : train.jsonl, val.jsonl, test.jsonl\n"
        f"\n➡️ Passe à l'onglet 'Entraîner un modèle' pour lancer le fine-tuning."
    )
    summary_table = [[len(chunks), len(dataset), len(deduped), len(train), len(val), len(test)]]
    yield log, summary_table

print("✅ Pipeline de génération de dataset prêt.")

## Étape 5 — Registre des modèles + fonctions génériques de fine-tuning

In [ ]:
MODEL_REGISTRY = {
    "mT5-small": {"hf_id": "google/mt5-small", "type": "seq2seq", "full_ft_lr": 1e-4, "peft_lr": 1e-3},
    "mT5-base": {"hf_id": "google/mt5-base", "type": "seq2seq", "full_ft_lr": 5e-5, "peft_lr": 1e-3},
    "Qwen2.5-1.5B-Instruct": {"hf_id": "Qwen/Qwen2.5-1.5B-Instruct", "type": "causal", "full_ft_lr": 1e-5, "peft_lr": 2e-4},
    "Phi-3.5-mini-instruct": {"hf_id": "microsoft/Phi-3.5-mini-instruct", "type": "causal", "full_ft_lr": 1e-5, "peft_lr": 2e-4},
}
METHODS = ["QLoRA", "LoRA", "Full Fine-Tuning"]
KNOWN_OOM = {("Phi-3.5-mini-instruct", "Full Fine-Tuning")}

from transformers import (
    AutoTokenizer as _AT, AutoModelForSeq2SeqLM, AutoModelForCausalLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments, BitsAndBytesConfig, BertTokenizer
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import evaluate, gc, time

# ⚠️ Correctif compatibilité : `bert_score` appelle tokenizer.build_inputs_with_special_tokens(),
# une méthode supprimée dans transformers 5.x (plus présente ni sur BertTokenizer ni BertTokenizerFast).
# Sans ce patch, evaluate_ft_model plante avec AttributeError au moment du calcul BERTScore.
if not hasattr(BertTokenizer, "build_inputs_with_special_tokens"):
    def _build_inputs_with_special_tokens(self, token_ids_0, token_ids_1=None):
        cls, sep = [self.cls_token_id], [self.sep_token_id]
        if token_ids_1 is None:
            return cls + token_ids_0 + sep
        return cls + token_ids_0 + sep + token_ids_1 + sep
    BertTokenizer.build_inputs_with_special_tokens = _build_inputs_with_special_tokens

rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)


def load_model_and_tokenizer(model_key, method):
    info = MODEL_REGISTRY[model_key]
    hf_id, model_type = info["hf_id"], info["type"]
    tokenizer = _AT.from_pretrained(hf_id)
    if model_type == "causal" and tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    load_class = AutoModelForSeq2SeqLM if model_type == "seq2seq" else AutoModelForCausalLM

    if method == "QLoRA":
        model = load_class.from_pretrained(hf_id, quantization_config=bnb_config, device_map="auto")
        model = prepare_model_for_kbit_training(model)
    elif method == "LoRA":
        model = load_class.from_pretrained(hf_id, torch_dtype=torch.bfloat16, device_map="auto")
        # Gradient checkpointing pour limiter la VRAM sur les modèles causaux (Qwen/Phi) sur T4 16 Go
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()
        model.config.use_cache = False
    else:
        model = load_class.from_pretrained(hf_id, torch_dtype=torch.bfloat16, device_map="auto")
        # ⚠️ NE PAS faire model.config.tie_word_embeddings = False ici : pour mT5/T5, ça casse le
        # rechargement du meilleur checkpoint (load_best_model_at_end=True). Les poids partagés
        # (shared.weight) ne sont sauvegardés qu'une fois ; si on "détie" après coup, le rechargement
        # considère encoder.embed_tokens.weight / decoder.embed_tokens.weight comme manquants et les
        # réinitialise au hasard → loss qui explose (~30 au lieu de ~4-6). On garde le tying par défaut.
        model.gradient_checkpointing_enable()
        model.config.use_cache = False

    if method in ("QLoRA", "LoRA"):
        target_modules = ["q", "v"] if model_type == "seq2seq" else ["q_proj", "v_proj", "k_proj", "o_proj"]
        task_type = TaskType.SEQ_2_SEQ_LM if model_type == "seq2seq" else TaskType.CAUSAL_LM
        lc = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=target_modules, task_type=task_type)
        model = get_peft_model(model, lc)
    return model, tokenizer, model_type


def prepare_ft_dataset(tokenizer, model_type):
    if model_type == "seq2seq":
        def preprocess(examples):
            inputs = ["répondre à la question: " + q for q in examples["question"]]
            mi = tokenizer(inputs, max_length=256, truncation=True, padding="max_length")
            labels = tokenizer(text_target=examples["answer"], max_length=256, truncation=True, padding="max_length")
            labels["input_ids"] = [[(t if t != tokenizer.pad_token_id else -100) for t in l] for l in labels["input_ids"]]
            mi["labels"] = labels["input_ids"]
            return mi
        train_ds = load_dataset("json", data_files="train.jsonl")["train"].map(preprocess, batched=True)
        val_ds = load_dataset("json", data_files="val.jsonl")["train"].map(preprocess, batched=True)
    else:
        def format_chat(ex):
            messages = [{"role": "user", "content": ex["question"]}, {"role": "assistant", "content": ex["answer"]}]
            return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}
        train_ds = load_dataset("json", data_files="train.jsonl")["train"].map(format_chat)
        val_ds = load_dataset("json", data_files="val.jsonl")["train"].map(format_chat)
    return train_ds, val_ds


def ask_model(model, tokenizer, model_type, question, max_new_tokens=100):
    if model_type == "seq2seq":
        input_text = "répondre à la question: " + question
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=256).to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4,
                                  repetition_penalty=1.3, no_repeat_ngram_size=3, early_stopping=True)
        text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return re.sub(r"<extra_id_\d+>", "", text).strip()
    else:
        messages = [{"role": "user", "content": question}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def evaluate_ft_model(model, tokenizer, model_type, test_data):
    predictions, references = [], []
    for sample in test_data:
        predictions.append(ask_model(model, tokenizer, model_type, sample["question"]))
        references.append(sample["answer"])
    rouge_res = rouge.compute(predictions=predictions, references=references)
    bert_res = bertscore.compute(predictions=predictions, references=references, lang="fr")
    return {"rouge1": rouge_res["rouge1"], "rougeL": rouge_res["rougeL"],
            "bertscore_f1": sum(bert_res["f1"]) / len(bert_res["f1"])}

print("✅ Fonctions de fine-tuning prêtes.")

## Étape 6 — Pipeline d'entraînement (générateur avec suivi de progression)

In [ ]:
_prev_model_ref = None
_prev_trainer_ref = None

def run_training_pipeline(model_key, method, n_epochs):
    global _prev_model_ref, _prev_trainer_ref

    if not (os.path.exists("train.jsonl") and os.path.exists("val.jsonl") and os.path.exists("test.jsonl")):
        yield "⚠️ Aucun dataset trouvé. Va d'abord dans l'onglet 'Générer le dataset'.", None
        return

    if (model_key, method) in KNOWN_OOM:
        yield f"⚠️ {model_key} + {method} échoue par manque de mémoire sur un GPU T4 16 Go. Choisis une autre combinaison.", None
        return

    log = f"🚀 Lancement : {model_key} | {method} | {n_epochs} epochs\n"
    yield log, None

    # Libère la VRAM du run précédent (l'ancien nettoyage via dir()/globals() ne fonctionnait pas :
    # dir() dans une fonction ne liste que les noms locaux déjà liés, donc la condition n'était jamais vraie)
    if _prev_trainer_ref is not None:
        del _prev_trainer_ref
        _prev_trainer_ref = None
    if _prev_model_ref is not None:
        del _prev_model_ref
        _prev_model_ref = None
    gc.collect()
    torch.cuda.empty_cache()

    log += "📥 Chargement du modèle et du tokenizer...\n"
    yield log, None
    model_current, tokenizer_current, model_type = load_model_and_tokenizer(model_key, method)

    log += "📚 Préparation du dataset...\n"
    yield log, None
    train_ds, val_ds = prepare_ft_dataset(tokenizer_current, model_type)
    test_data = [json.loads(l) for l in open("test.jsonl", encoding="utf-8")]

    info = MODEL_REGISTRY[model_key]
    lr = info["full_ft_lr"] if method == "Full Fine-Tuning" else info["peft_lr"]
    output_dir = f"./results_{model_key}_{method}".replace(" ", "_")

    common_args = dict(output_dir=output_dir, num_train_epochs=n_epochs, learning_rate=lr, logging_steps=20,
                        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
                        load_best_model_at_end=True, metric_for_best_model="eval_loss",
                        greater_is_better=False, report_to="none", seed=42)

    if model_type == "seq2seq":
        bs, ga = (2, 2) if method == "Full Fine-Tuning" else (4, 1)
        args = Seq2SeqTrainingArguments(per_device_train_batch_size=bs, gradient_accumulation_steps=ga,
                                         predict_with_generate=True, **common_args)
        trainer_current = Seq2SeqTrainer(model=model_current, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                                          processing_class=tokenizer_current)
    else:
        if method == "Full Fine-Tuning":
            args = SFTConfig(per_device_train_batch_size=1, gradient_accumulation_steps=8,
                              optim="paged_adamw_8bit", dataset_text_field="text", max_length=512, **common_args)
        else:
            args = SFTConfig(per_device_train_batch_size=2, gradient_accumulation_steps=4,
                              dataset_text_field="text", max_length=512, **common_args)
        # ⚠️ LE BUG ÉTAIT ICI : sans processing_class explicite, SFTTrainer (trl récent) tente
        # AutoProcessor.from_pretrained(model.config._name_or_path) automatiquement. Qwen et Phi
        # n'ont pas de processor_config.json (ce ne sont pas des modèles multimodaux) → ça plantait.
        # mT5 passe par Seq2SeqTrainer (pas SFTTrainer) donc n'était pas affecté, d'où l'écart observé.
        trainer_current = SFTTrainer(model=model_current, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                                      processing_class=tokenizer_current)

    log += f"🏋️ Entraînement en cours ({n_epochs} epochs)...\n"
    yield log, None

    start = time.time()
    trainer_current.train()
    elapsed = (time.time() - start) / 60

    log += f"✅ Entraînement terminé en {elapsed:.1f} min\n📊 Évaluation sur le test set...\n"
    yield log, None

    results = evaluate_ft_model(model_current, tokenizer_current, model_type, test_data)
    results.update({"training_time_min": elapsed, "model": model_key, "method": method, "epochs": n_epochs})

    results_filename = f"results_{model_key}_{method}.json".replace(" ", "_")
    with open(results_filename, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    save_to_drive([results_filename])

    # Sauvegarde le modèle entraîné (poids + tokenizer + métadonnées) pour l'onglet "Poser une question"
    final_dir = f"trained_models/{model_key}_{method}".replace(" ", "_")
    os.makedirs(final_dir, exist_ok=True)
    trainer_current.save_model(final_dir)
    tokenizer_current.save_pretrained(final_dir)
    with open(os.path.join(final_dir, "meta.json"), "w", encoding="utf-8") as f:
        json.dump({"model_key": model_key, "method": method, "model_type": model_type,
                    "hf_id": info["hf_id"]}, f, ensure_ascii=False)
    save_dir_to_drive(final_dir)

    log += (f"\n🎯 RÉSULTATS : ROUGE-1={results['rouge1']:.3f} | ROUGE-L={results['rougeL']:.3f} | "
            f"BERTScore={results['bertscore_f1']:.3f}\n"
            f"💾 Modèle sauvegardé dans {final_dir}/ (disponible dans l'onglet 'Poser une question')\n")

    table = [[model_key, method, n_epochs, round(results["rouge1"], 3),
              round(results["rougeL"], 3), round(results["bertscore_f1"], 3), round(elapsed, 1)]]

    _prev_model_ref = model_current
    _prev_trainer_ref = trainer_current

    yield log, table

print("✅ Pipeline d'entraînement prêt.")

### 🔎 Fonctions pour l'onglet "Poser une question" (charger et interroger un modèle déjà entraîné)

In [ ]:
import glob
from peft import PeftModel

TRAINED_MODELS_DIR = "trained_models"
_inference_cache = {}  # évite de recharger le même modèle à chaque question


def list_trained_models():
    """Scanne trained_models/*/meta.json et retourne les combinaisons disponibles pour le Dropdown."""
    choices = []
    for meta_path in sorted(glob.glob(os.path.join(TRAINED_MODELS_DIR, "*", "meta.json"))):
        folder = os.path.dirname(meta_path)
        with open(meta_path, encoding="utf-8") as f:
            meta = json.load(f)
        label = f"{meta['model_key']} + {meta['method']}"
        choices.append((label, folder))
    return choices


def load_trained_model_for_inference(folder):
    if folder in _inference_cache:
        return _inference_cache[folder]

    with open(os.path.join(folder, "meta.json"), encoding="utf-8") as f:
        meta = json.load(f)
    hf_id, model_type, method = meta["hf_id"], meta["model_type"], meta["method"]
    load_class = AutoModelForSeq2SeqLM if model_type == "seq2seq" else AutoModelForCausalLM
    tokenizer = _AT.from_pretrained(folder)

    if method in ("LoRA", "QLoRA"):
        # Pour LoRA/QLoRA, seul l'adaptateur est sauvegardé : on recharge le modèle de base
        # (en bf16, sans quantization : suffisant et plus simple pour de l'inférence) puis l'adaptateur.
        base_model = load_class.from_pretrained(hf_id, torch_dtype=torch.bfloat16, device_map="auto")
        model = PeftModel.from_pretrained(base_model, folder)
    else:
        # Full Fine-Tuning : les poids complets ont été sauvegardés directement dans folder
        model = load_class.from_pretrained(folder, torch_dtype=torch.bfloat16, device_map="auto")

    model.eval()
    _inference_cache[folder] = (model, tokenizer, model_type)
    return model, tokenizer, model_type


def answer_question(folder, question):
    if not folder:
        return "⚠️ Choisis d'abord un modèle entraîné dans la liste (entraîne-en un dans l'onglet 2 si la liste est vide)."
    if not question or not question.strip():
        return "⚠️ Tape une question."
    model, tokenizer, model_type = load_trained_model_for_inference(folder)
    return ask_model(model, tokenizer, model_type, question.strip())

print("✅ Fonctions d'inférence prêtes.")

## Étape 7 — Interface web Gradio (3 onglets)

In [ ]:
import gradio as gr

with gr.Blocks(title="Pipeline QA génératif") as demo:
    gr.Markdown("# 🧠 Pipeline Data Ingestion + Fine-Tuning QA génératif")

    with gr.Tab("1️⃣ Générer le dataset"):
        gr.Markdown("Upload un document (PDF, DOCX, TXT, CSV, image ou JSON) pour créer le dataset Q/A.")
        file_input = gr.File(label="Document source", file_types=[".pdf", ".docx", ".txt", ".csv", ".png", ".jpg", ".jpeg", ".json"])
        generate_btn = gr.Button("📄 Générer le dataset", variant="primary")
        gen_log = gr.Textbox(label="Journal", lines=12, interactive=False)
        gen_summary = gr.Dataframe(
            headers=["Chunks", "Q/A brutes", "Q/A finales", "Train", "Val", "Test"],
            label="Résumé du dataset généré",
        )

        def on_generate(file_obj):
            if file_obj is None:
                yield "⚠️ Upload d'abord un document.", None
                return
            for log, table in build_dataset_pipeline(file_obj.name):
                yield log, table

        generate_btn.click(fn=on_generate, inputs=[file_input], outputs=[gen_log, gen_summary])

    with gr.Tab("2️⃣ Entraîner un modèle"):
        gr.Markdown("Choisis un modèle et une méthode de fine-tuning (nécessite un dataset déjà généré).")
        with gr.Row():
            model_choice = gr.Dropdown(choices=list(MODEL_REGISTRY.keys()), value="mT5-base", label="Modèle")
            method_choice = gr.Dropdown(choices=METHODS, value="QLoRA", label="Méthode")
            epochs_choice = gr.Slider(minimum=1, maximum=20, value=5, step=1, label="Epochs")
        train_btn = gr.Button("🚀 Lancer l'entraînement", variant="primary")
        train_log = gr.Textbox(label="Journal d'exécution", lines=12, interactive=False)
        train_result = gr.Dataframe(
            headers=["Modèle", "Méthode", "Epochs", "ROUGE-1", "ROUGE-L", "BERTScore F1", "Temps (min)"],
            label="Résultat de ce run",
        )

        def on_train(model_key, method, n_epochs):
            for log, table in run_training_pipeline(model_key, method, n_epochs):
                yield log, table

        train_btn.click(fn=on_train, inputs=[model_choice, method_choice, epochs_choice], outputs=[train_log, train_result])

    with gr.Tab("3️⃣ Poser une question"):
        gr.Markdown("Choisis un modèle déjà entraîné (onglet précédent) et pose-lui une question en langage naturel.")
        with gr.Row():
            trained_model_choice = gr.Dropdown(choices=[], label="Modèle entraîné", interactive=True, scale=4)
            refresh_btn = gr.Button("🔄 Rafraîchir la liste", scale=1)
        question_input = gr.Textbox(label="Ta question", lines=2, placeholder="Ex : Que dois-je faire en cas de conflit d'intérêts ?")
        ask_btn = gr.Button("💬 Obtenir une réponse", variant="primary")
        answer_output = gr.Textbox(label="Réponse du modèle", lines=6, interactive=False)

        def refresh_model_list():
            choices = list_trained_models()
            return gr.Dropdown(choices=choices, value=(choices[0][1] if choices else None))

        demo.load(fn=refresh_model_list, outputs=[trained_model_choice])
        refresh_btn.click(fn=refresh_model_list, outputs=[trained_model_choice])
        ask_btn.click(fn=answer_question, inputs=[trained_model_choice, question_input], outputs=[answer_output])

demo.launch(share=True, debug=True)

## Étape 8 — Consolider tous les résultats obtenus (à exécuter après avoir arrêté l'interface)

In [ ]:
import glob
import pandas as pd

all_results = []
for path in glob.glob("results_*.json"):
    try:
        with open(path, encoding="utf-8") as f:
            r = json.load(f)
        if "model" in r and "method" in r:
            all_results.append(r)
    except (json.JSONDecodeError, KeyError):
        continue

if all_results:
    df_all = pd.DataFrame(all_results)[["model", "method", "epochs", "rouge1", "rougeL", "bertscore_f1", "training_time_min"]]
    df_all = df_all.sort_values("rouge1", ascending=False).reset_index(drop=True)
    display(df_all)
    df_all.to_csv("tableau_final_toutes_combinaisons.csv", index=False)
    save_to_drive(["tableau_final_toutes_combinaisons.csv"])
else:
    print("Aucun résultat trouvé. Lance d'abord au moins une combinaison via l'interface.")